In [ ]:
import os, sys

notebook_path = os.getcwd()
root = os.path.dirname(notebook_path)
sys.path.insert(0, root)

from embed_sim import spadecl, rdiis, ssdmet
from pyscf import gto, scf, mp, dft
import time
import scipy
import numpy as np

bath_option = 0
energys = {'b3lyp':{}, 'hf':{}}
xcs = ['b3lyp', 'hf']
spins = [1, 3]
for xc in xcs:
    for spin in spins:
    
        title = 'CoSH4'
        def get_mol(dihedral):
             mol = gto.M(atom = '''
                        Co             
                        S                  1            2.30186590
                        S                  1            2.30186590    2            109.47122060
                        S                  1            2.30186590    3            109.47122065    2            -120.00000001                  0
                        S                  1            2.30186590    4            109.47122060    3            120.00000001                   0
                        H                  2            1.30714645    1            109.47121982    4            '''+str(-60-dihedral)+'''      0
                        H                  4            1.30714645    1            109.47121982    3            '''+str(60+dihedral)+'''       0
                        H                  5            1.30714645    1            109.47121982    4            '''+str(-180+dihedral)+'''     0
                        H                  3            1.30714645    1            109.47121982    4            '''+str(60-dihedral)+'''       0
             ''',
             basis={'default':'def2tzvp','s':'6-31G*','H':'6-31G*'}, symmetry=0 ,spin = spin,charge = -2,verbose= 4)
        
             return mol
        
        energy = []
        for i in range(0, 181, 30):
            start = time.time()  # 记录开始时间
            chk_fname = title + '_dihedral' + str(i) + '_ls.chk'
            mol = get_mol(i)
        
            mf = scf.rohf.ROHF(mol).density_fit().x2c()
            mf.chkfile = chk_fname
            mf.diis = rdiis.RDIIS(rdiis_prop='dS', imp_idx=mol.search_ao_label(['Co.*d']), power=0.2)
            mf.init_guess = 'atom'
            mf.level_shift = .2
            mf.diis_space = 16
            mf.max_cycle = 10000
            mf.max_memory = 4000
            mf.kernel()
            end = time.time()    # 记录结束时间
            print(f"全电子ROHF耗时: {end - start:.3f} 秒")
            assert(mf.converged)
        
            start = time.time()  # 记录开始时间
            # mydmet = spadecl.SPADECL(mf, title=title, imp_idx=['Co.*'], es_natorb=True, bath_option=bath_option)
            # mydmet.build(xc=xc)
            mydmet = ssdmet.SSDMET(mf, title=title, imp_idx=['Co.1s', 'Co.2s', 'Co.2p', 'Co.3s', 'Co.3p', 'Co.3d', 'Co.4s', 'Co.4p', 'Co.4d'], es_natorb=True)
            mydmet.build(restore_imp=True)
            end = time.time()    # 记录结束时间
            print(f"SPADECL耗时: {end - start:.3f} 秒")
        
            start = time.time()  # 记录开始时间
            es_mf = mydmet.es_mf
            es_mf.max_cycle = 1000
            print(es_mf.e_tot)
            es_mf.kernel()
            
            es_mp = mp.ump2.UMP2(es_mf.to_uhf())
            es_mp.max_cycle = 1000
            es_mp.kernel()
            es_mp2_corr = es_mp.e_corr
            print("es_mp.e_corr:", es_mp.e_corr)
            end = time.time()    # 记录结束时间
            print(f"es_mp2耗时: {end - start:.3f} 秒")
        
            # def get_es_dm(mydmet, es_mf):
            #     es_dm_active = es_mf.make_rdm1()
            #     es_dma = mydmet.es_orb @ es_dm_active[0] @ mydmet.es_orb.T.conj()
            #     es_dmb = mydmet.es_orb @ es_dm_active[1] @ mydmet.es_orb.T.conj()
            #     return (es_dma, es_dmb)
            
            # es_dm = get_es_dm(mydmet, es_mf)
            # es_dm = es_dm[0] + es_dm[1]
        
            # e_dm_corr = np.einsum("ij,ji->", mydmet.veff_emb, es_dm-(mydmet.es_dm0[0]+mydmet.es_dm0[1]))
            # e_es = mydmet.get_e_mf()
            
            # e_CL = e_es + mydmet.e_fo + mydmet.e_cross + mydmet.e_nuc + e_dm_corr + es_mp.e_corr
            mydmet.get_corr(xc=xc)
            e_CL = es_mp.e_tot + mydmet.e_fo + mydmet.e_nuc
            # print(e_es, mydmet.e_fo, mydmet.e_cross, mydmet.e_nuc, e_dm_corr)
            print(f"e_CL{i}:", e_CL)
            energy.append(e_CL)
        print(energy)

        energys[xc][spin] = energy.copy()



System: uname_result(system='Linux', node='localhost', release='4.4.0-26100-Microsoft', version='#8972-Microsoft Fri Jan 01 08:00:00 PST 2016', machine='x86_64')  Threads 8
Python 3.10.12 (main, Feb  4 2025, 14:57:36) [GCC 11.4.0]
numpy 1.24.2  scipy 1.15.3  h5py 3.13.0
Date: Mon Aug 24 12:10:58 2026
PySCF version 2.9.0
PySCF path  /home/soda/.local/lib/python3.10/site-packages/pyscf

[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 9
[INPUT] num. electrons = 97
[INPUT] charge = -2
[INPUT] spin (= nelec alpha-beta = 2S) = 1
[INPUT] symmetry 0 subgroup None
[INPUT] Mole.unit = angstrom
[INPUT] Symbol           X                Y                Z      unit          X                Y                Z       unit  Magmom
[INPUT]  1 Co     0.000000000000   0.000000000000   0.000000000000 AA    0.000000000000   0.000000000000   0.000000000000 Bohr   0.0
[INPUT]  2 S      2.301865900000   0.000000000000   0.000000000000 AA    4.349896126475   0.000000000000   0.000000000000 Bo

/home/soda/.local/lib/python3.10/site-packages/pyscf/dft/libxc.py:512: UserWarning: Since PySCF-2.3, B3LYP (and B3P86) are changed to the VWN-RPA variant, corresponding to the original definition by Stephens et al. (issue 1480) and the same as the B3LYP functional in Gaussian. To restore the VWN5 definition, you can put the setting "B3LYP_WITH_VWN5 = True" in pyscf_conf.py
  warnings.warn('Since PySCF-2.3, B3LYP (and B3P86) are changed to the VWN-RPA variant, '


******** <class 'pyscf.df.df.DF'> ********
auxbasis = auxmol.basis = {'S': 'def2-tzvp-jkfit', 'Co': 'def2-tzvp-jkfit', 'H': 'cc-pvdz-jkfit', 's': 'cc-pvdz-jkfit'}
max_memory = 4000
Default auxbasis def2-tzvp-jkfit is used for S def2tzvp
Default auxbasis def2-tzvp-jkfit is used for Co def2tzvp
Default auxbasis cc-pvdz-jkfit is used for H 6-31G*
Default auxbasis cc-pvdz-jkfit is used for s 6-31G*
-3.0638221906664
-2125.1127029584977 -2093.0003124935442 -3673.428525186263 -2990.373492264839
e_CL0: -2989.0976678088537
System: uname_result(system='Linux', node='localhost', release='4.4.0-26100-Microsoft', version='#8972-Microsoft Fri Jan 01 08:00:00 PST 2016', machine='x86_64')  Threads 8
Python 3.10.12 (main, Feb  4 2025, 14:57:36) [GCC 11.4.0]
numpy 1.24.2  scipy 1.15.3  h5py 3.13.0
Date: Mon Aug 24 12:11:14 2026
PySCF version 2.9.0
PySCF path  /home/soda/.local/lib/python3.10/site-packages/pyscf

[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 9
[INPUT] num. electrons = 

In [2]:
for xc in xcs:
    for spin in spins:
        print(energys[xc][spin])

[-2989.5120039062967, -2989.5096302105185, -2989.511665846226, -2989.50988815272, -2989.5059691327174, -2989.5138671169757, -2989.5153827352274]
[-2989.594138852555, -2989.5962070517253, -2989.595929844348, -2989.5961667531187, -2989.5995136335187, -2989.5986147880435, -2989.5941255839]
[-2986.616107918073, -2986.6101096306325, -2986.6112078602737, -2986.610249569951, -2986.6087362403705, -2986.6149085811658, -2986.615613530655]
[-2986.717583295994, -2986.7192415268787, -2986.7183159892706, -2986.7187918643, -2986.722821012526, -2986.721620784313, -2986.716542798685]


In [3]:
# no corr
for xc in xcs:
    for spin in spins:
        print(energys[xc][spin])

[-2989.413477321649, -2989.4050186808167, -2989.40654777601, -2989.405085405626, -2989.40262254024, -2989.4078397647454, -2989.408411512889]
[-2989.5012430426495, -2989.503600272373, -2989.5034679336472, -2989.50353103191, -2989.506520962349, -2989.505841452929, -2989.5016830397403]
[-2986.616107902866, -2986.6101095468543, -2986.611208045995, -2986.6102495306995, -2986.6087356147377, -2986.61491112216, -2986.615613225028]
[-2986.7175832944595, -2986.7192415270706, -2986.718315989289, -2986.7187918641994, -2986.7228210122653, -2986.721620784418, -2986.716542799713]


In [3]:
import os, sys

notebook_path = os.getcwd()
root = os.path.dirname(notebook_path)
sys.path.insert(0, root)

from embed_sim import spadecl, rdiis
from pyscf import gto, scf, mp, dft
import time
import scipy
import numpy as np

bath_option = 0

energysas = {}
spins = [1, 3]
for spin in spins:
    
    title = 'CoSH4'
    def get_mol(dihedral):
         mol = gto.M(atom = '''
                    Co             
                    S                  1            2.30186590
                    S                  1            2.30186590    2            109.47122060
                    S                  1            2.30186590    3            109.47122065    2            -120.00000001                  0
                    S                  1            2.30186590    4            109.47122060    3            120.00000001                   0
                    H                  2            1.30714645    1            109.47121982    4            '''+str(-60-dihedral)+'''      0
                    H                  4            1.30714645    1            109.47121982    3            '''+str(60+dihedral)+'''       0
                    H                  5            1.30714645    1            109.47121982    4            '''+str(-180+dihedral)+'''     0
                    H                  3            1.30714645    1            109.47121982    4            '''+str(60-dihedral)+'''       0
         ''',
         basis={'default':'def2tzvp','s':'6-31G*','H':'6-31G*'}, symmetry=0 ,spin = spin,charge = -2,verbose= 4)
    
         return mol
    
    energy_as = []
    for i in range(0, 181, 30):
        start = time.time()  # 记录开始时间
        chk_fname = title + '_dihedral' + str(i) + '_ls.chk'
        mol = get_mol(i)
    
        mf = scf.rohf.ROHF(mol).density_fit().x2c()
        mf.chkfile = chk_fname
        mf.diis = rdiis.RDIIS(rdiis_prop='dS', imp_idx=mol.search_ao_label(['Co.*d']), power=0.2)
        mf.init_guess = 'atom'
        mf.level_shift = .2
        mf.diis_space = 16
        mf.max_cycle = 10000
        mf.max_memory = 4000
        mf.kernel()
        end = time.time()    # 记录结束时间
        print(f"全电子ROHF耗时: {end - start:.3f} 秒")
        assert(mf.converged)
    
        start = time.time()  # 记录开始时间
        mymp = mp.ump2.UMP2(mf.to_uhf())
        mymp.max_cycle = 1000
        mymp.kernel()
        mymp2_corr = mymp.e_corr
        print("as_mp.e_corr:", mymp.e_corr)
        end = time.time()    # 记录结束时间
        print(f"as_mp2耗时: {end - start:.3f} 秒")
    
        energy_as.append(mymp.e_corr + mf.e_tot)
    print(energy_as)
    energysas[spin] = energy_as.copy()


System: uname_result(system='Linux', node='localhost', release='4.4.0-26100-Microsoft', version='#8972-Microsoft Fri Jan 01 08:00:00 PST 2016', machine='x86_64')  Threads 8
Python 3.10.12 (main, Feb  4 2025, 14:57:36) [GCC 11.4.0]
numpy 1.24.2  scipy 1.15.3  h5py 3.13.0
Date: Sun Aug 23 23:09:18 2026
PySCF version 2.9.0
PySCF path  /home/soda/.local/lib/python3.10/site-packages/pyscf

[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 9
[INPUT] num. electrons = 97
[INPUT] charge = -2
[INPUT] spin (= nelec alpha-beta = 2S) = 1
[INPUT] symmetry 0 subgroup None
[INPUT] Mole.unit = angstrom
[INPUT] Symbol           X                Y                Z      unit          X                Y                Z       unit  Magmom
[INPUT]  1 Co     0.000000000000   0.000000000000   0.000000000000 AA    0.000000000000   0.000000000000   0.000000000000 Bohr   0.0
[INPUT]  2 S      2.301865900000   0.000000000000   0.000000000000 AA    4.349896126475   0.000000000000   0.000000000000 Bo

In [4]:
for spin in spins:
    print(energysas[spin])

[-2987.2280364297985, -2987.227617091075, -2987.229852885487, -2987.2281151992875, -2987.224240478578, -2987.2332399553557, -2987.2352360884024]
[-2987.3173315141767, -2987.319419912794, -2987.318836198993, -2987.319133604226, -2987.3230342420675, -2987.321967777273, -2987.3167050971592]


In [6]:
print((np.array(energys['hf'][3]) - np.array(energys['hf'][1])) - (np.array(energysas[3]) - np.array(energysas[1])))

[-0.01218029 -0.01732907 -0.01812482 -0.01752389 -0.01529101 -0.01798438
 -0.01946026]


In [8]:
print((np.array(energys['b3lyp'][3]) - np.array(energys['b3lyp'][1])) - (np.array(energysas[3]) - np.array(energysas[1])))

[0.00716014 0.00522598 0.00471932 0.0047398  0.00524926 0.00398015
 0.00272616]


In [10]:
print((np.array(energys['hf'][3]) - np.array(energys['hf'][1])) + (np.array([-2987.2280364297985, -2987.227617091075, -2987.229852885487, -2987.2281151992875, -2987.224240478578, -2987.2332399553557, -2987.2352360884024]) - np.array([-2987.3173315141767, -2987.319419912794, -2987.318836198993, -2987.319133604226, -2987.3230342420675, -2987.321967777273, -2987.3167050971592])))

[-0.01218028 -0.01732927 -0.01812478 -0.01752414 -0.01529114 -0.01798732
 -0.0194612 ]


In [11]:
print((np.array(energys['b3lyp'][3]) - np.array(energys['b3lyp'][1])) + (np.array([-2987.2280364297985, -2987.227617091075, -2987.229852885487, -2987.2281151992875, -2987.224240478578, -2987.2332399553557, -2987.2352360884024]) - np.array([-2987.3173315141767, -2987.319419912794, -2987.318836198993, -2987.319133604226, -2987.3230342420675, -2987.321967777273, -2987.3167050971592])))

[ 0.00152926 -0.00677704 -0.00793715 -0.00742799 -0.00510439 -0.00927758
 -0.01181004]


In [ ]:
print((np.array(energys['hf'][3]) - np.array(energys['hf'][1])) + (np.array([-2987.2280364297985, -2987.227617091075, -2987.229852885487, -2987.2281151992875, -2987.224240478578, -2987.2332399553557, -2987.2352360884024]) - np.array([-2987.3173315141767, -2987.319419912794, -2987.318836198993, -2987.319133604226, -2987.3230342420675, -2987.321967777273, -2987.3167050971592])))

In [ ]:
print((np.array(energys['b3lyp'][3]) - np.array(energys['b3lyp'][1])) + (np.array([-2987.2280364297985, -2987.227617091075, -2987.229852885487, -2987.2281151992875, -2987.224240478578, -2987.2332399553557, -2987.2352360884024]) - np.array([-2987.3173315141767, -2987.319419912794, -2987.318836198993, -2987.319133604226, -2987.3230342420675, -2987.321967777273, -2987.3167050971592])))

In [4]:
A0 = np.array(energy)-np.array([-2987.31733151,-2987.31941991,-2987.3188362,-2987.3191336,-2987.32303424,-2987.32196778,-2987.3167051])
print((A0-A0[0])*27.2113863)
print((A0-A0[0])*27.2113863*23.06)

[0.         0.01170548 0.02100707 0.01615054 0.01265364 0.01629368
 0.01126787]
[0.         0.26992828 0.48442307 0.37243154 0.29179298 0.37573216
 0.25983708]


In [9]:
print(np.array(energy))
print(np.array(energy_as))

[-2981.50961775 -2981.50632517 -2981.51264911 -2981.51685854
 -2981.51038698 -2981.50185579 -2981.52104069]
[-2987.31733151 -2987.31941991 -2987.3188362  -2987.3191336
 -2987.32303424 -2987.32196778 -2987.3167051 ]


In [5]:
import os, sys

notebook_path = os.getcwd()
root = os.path.dirname(notebook_path)
sys.path.insert(0, root)

from embed_sim import spadecl, rdiis
from pyscf import gto, scf, mp, dft
import time
import scipy
import numpy as np

bath_option = 1

title = 'CoSH4'
def get_mol(dihedral):
     mol = gto.M(atom = '''
                Co             
                S                  1            2.30186590
                S                  1            2.30186590    2            109.47122060
                S                  1            2.30186590    3            109.47122065    2            -120.00000001                  0
                S                  1            2.30186590    4            109.47122060    3            120.00000001                   0
                H                  2            1.30714645    1            109.47121982    4            '''+str(-60-dihedral)+'''      0
                H                  4            1.30714645    1            109.47121982    3            '''+str(60+dihedral)+'''       0
                H                  5            1.30714645    1            109.47121982    4            '''+str(-180+dihedral)+'''     0
                H                  3            1.30714645    1            109.47121982    4            '''+str(60-dihedral)+'''       0
     ''',
     basis={'default':'def2tzvp','s':'6-31G*','H':'6-31G*'}, symmetry=0 ,spin = 3,charge = -2,verbose= 4)

     return mol

energy1 = []
for i in range(0, 181, 30):
    start = time.time()  # 记录开始时间
    chk_fname = title + '_dihedral' + str(i) + '_ls.chk'
    mol = get_mol(i)

    mf = scf.rohf.ROHF(mol).density_fit().x2c()
    mf.chkfile = chk_fname
    mf.diis = rdiis.RDIIS(rdiis_prop='dS', imp_idx=mol.search_ao_label(['Co.*d']), power=0.2)
    mf.init_guess = 'atom'
    mf.level_shift = .2
    mf.diis_space = 16
    mf.max_cycle = 10000
    mf.max_memory = 4000
    mf.kernel()
    end = time.time()    # 记录结束时间
    print(f"全电子ROHF耗时: {end - start:.3f} 秒")
    assert(mf.converged)

    start = time.time()  # 记录开始时间
    mydmet = spadecl.SPADECL(mf, title=title, imp_idx=['Co.1s', 'Co.2s', 'Co.3s', 'Co.2p', 'Co.3p', 'Co.3d', 'S.2p', 'S.1s', 'S.2s'], es_natorb=True, bath_option=bath_option)
    mydmet.build()
    end = time.time()    # 记录结束时间
    print(f"SPADECL耗时: {end - start:.3f} 秒")
    print("mycas.fo_ene:", mydmet.fo_ene)

    start = time.time()  # 记录开始时间
    es_mf = mydmet.es_mf
    es_mf.max_cycle = 1000
    es_mf.kernel()
    
    es_mp = mp.ump2.UMP2(es_mf.to_uhf())
    es_mp.max_cycle = 1000
    es_mp.kernel()
    es_mp2_corr = es_mp.e_corr
    print("es_mp.e_corr:", es_mp.e_corr)
    end = time.time()    # 记录结束时间
    print(f"es_mp2耗时: {end - start:.3f} 秒")

    def get_es_dm(mydmet, es_mf, full=0):
        es_dm_active = es_mf.make_rdm1()
        es_dma_mo = scipy.linalg.block_diag(np.eye(mydmet.nfo)*full, es_dm_active[0], np.eye(mydmet.nfv)*0) 
        es_dmb_mo = scipy.linalg.block_diag(np.eye(mydmet.nfo)*full, es_dm_active[1], np.eye(mydmet.nfv)*0) 
        es_orb = np.hstack((mydmet.fo_orb, mydmet.es_orb, mydmet.fv_orb))
        es_dma = es_orb @ es_dma_mo @ es_orb.T.conj()
        es_dmb = es_orb @ es_dmb_mo @ es_orb.T.conj()
        return (es_dma, es_dmb)

    def get_e_g(mydmet, dm):
        dm = dm[0] + dm[1]
        fo_dm = mydmet.fo_orb @ mydmet.fo_orb.T.conj()*2
        vj, vk = mydmet.mf_or_cas.get_jk(mol=mydmet.mf_or_cas.mol, dm=fo_dm)

        g = vj - 0.5 * vk
        g_energy = np.einsum("ij,ij->", g, dm)
        return g_energy

    mydft = dft.UKS(mol)
    mydft.xc = 'b3lyp'

    es_dm = get_es_dm(mydmet, es_mf)
    es_dm_full = get_es_dm(mydmet, es_mf, 1)

    e_dft_corr = mydft.energy_tot(es_dm_full) - mydft.energy_tot(es_dm)
    e_g = get_e_g(mydmet, es_dm)

    e_CL = es_mp.e_tot+mf.energy_nuc()+e_dft_corr-e_g
    print(f"e_CL{i}:", e_CL)
    energy1.append(e_CL)
print(energy1)



System: uname_result(system='Linux', node='localhost', release='4.4.0-26100-Microsoft', version='#8737-Microsoft Fri Jan 01 08:00:00 PST 2016', machine='x86_64')  Threads 8
Python 3.10.12 (main, Feb  4 2025, 14:57:36) [GCC 11.4.0]
numpy 1.24.2  scipy 1.15.3  h5py 3.13.0
Date: Sat Aug 15 23:14:36 2026
PySCF version 2.9.0
PySCF path  /home/soda/.local/lib/python3.10/site-packages/pyscf

[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 9
[INPUT] num. electrons = 97
[INPUT] charge = -2
[INPUT] spin (= nelec alpha-beta = 2S) = 3
[INPUT] symmetry 0 subgroup None
[INPUT] Mole.unit = angstrom
[INPUT] Symbol           X                Y                Z      unit          X                Y                Z       unit  Magmom
[INPUT]  1 Co     0.000000000000   0.000000000000   0.000000000000 AA    0.000000000000   0.000000000000   0.000000000000 Bohr   0.0
[INPUT]  2 S      2.301865900000   0.000000000000   0.000000000000 AA    4.349896126475   0.000000000000   0.000000000000 Bo

In [6]:
A1 = np.array(energy1)-np.array(energy_as)
print((A1-A1[0])*27.2113863)
print((A1-A1[0])*27.2113863*23.06)

[ 0.         -0.2626048  -0.12654634 -0.23337373  0.06699785  0.39642598
  0.21700419]
[ 0.         -6.05566667 -2.91815858 -5.3815981   1.54497037  9.14158319
  5.00411656]


In [6]:
import os, sys

notebook_path = os.getcwd()
root = os.path.dirname(notebook_path)
sys.path.insert(0, root)

from embed_sim import ssdmet, rdiis
from pyscf import gto, scf, mp, dft
import time
import scipy
import numpy as np

bath_option={'ROMP2':0}

title = 'CoSH4'
def get_mol(dihedral):
     mol = gto.M(atom = '''
                Co             
                S                  1            2.30186590
                S                  1            2.30186590    2            109.47122060
                S                  1            2.30186590    3            109.47122065    2            -120.00000001                  0
                S                  1            2.30186590    4            109.47122060    3            120.00000001                   0
                H                  2            1.30714645    1            109.47121982    4            '''+str(-60-dihedral)+'''      0
                H                  4            1.30714645    1            109.47121982    3            '''+str(60+dihedral)+'''       0
                H                  5            1.30714645    1            109.47121982    4            '''+str(-180+dihedral)+'''     0
                H                  3            1.30714645    1            109.47121982    4            '''+str(60-dihedral)+'''       0
     ''',
     basis={'default':'def2tzvp','s':'6-31G*','H':'6-31G*'}, symmetry=0 ,spin = 3,charge = -2,verbose= 4)

     return mol

energy = []
for i in range(0, 181, 30):
    start = time.time()  # 记录开始时间
    chk_fname = title + '_dihedral' + str(i) + '_ls.chk'
    mol = get_mol(i)

    mf = scf.rohf.ROHF(mol).density_fit().x2c()
    mf.chkfile = chk_fname
    mf.diis = rdiis.RDIIS(rdiis_prop='dS', imp_idx=mol.search_ao_label(['Co.*d']), power=0.2)
    mf.init_guess = 'atom'
    mf.level_shift = .2
    mf.diis_space = 16
    mf.max_cycle = 10000
    mf.max_memory = 4000
    mf.kernel()
    end = time.time()    # 记录结束时间
    print(f"全电子ROHF耗时: {end - start:.3f} 秒")
    assert(mf.converged)

    start = time.time()  # 记录开始时间
    mydmet = ssdmet.SSDMET(mf, title=title, imp_idx=['Co.1s', 'Co.2s', 'Co.3s', 'Co.2p', 'Co.3p', 'Co.3d', 'S.2p'])
    mydmet.build(restore_imp=True)
    end = time.time()    # 记录结束时间
    print(f"SSDMET耗时: {end - start:.3f} 秒")
    print("mycas.fo_ene:", mydmet.fo_ene)

    start = time.time()  # 记录开始时间
    es_mf = mydmet.es_mf
    es_mf.max_cycle = 1000
    es_mf.kernel()
    
    es_mp = mp.ump2.UMP2(es_mf.to_uhf())
    es_mp.max_cycle = 1000
    es_mp.kernel()
    es_mp2_corr = es_mp.e_corr
    print("es_mp.e_corr:", es_mp.e_corr)
    end = time.time()    # 记录结束时间
    print(f"es_mp2耗时: {end - start:.3f} 秒")

    def get_es_dm(mydmet, es_mf, full=0):
        es_dm_active = es_mf.make_rdm1()
        es_dma_mo = scipy.linalg.block_diag(np.eye(mydmet.nfo)*full, es_dm_active[0], np.eye(mydmet.nfv)*0) 
        es_dmb_mo = scipy.linalg.block_diag(np.eye(mydmet.nfo)*full, es_dm_active[1], np.eye(mydmet.nfv)*0) 
        es_orb = np.hstack((mydmet.fo_orb, mydmet.es_orb, mydmet.fv_orb))
        es_dma = es_orb @ es_dma_mo @ es_orb.T.conj()
        es_dmb = es_orb @ es_dmb_mo @ es_orb.T.conj()
        return (es_dma, es_dmb)

    def get_e_g(mydmet, dm):
        dm = dm[0] + dm[1]
        fo_dm = mydmet.fo_orb @ mydmet.fo_orb.T.conj()*2
        vj, vk = mydmet.mf_or_cas.get_jk(mol=mydmet.mf_or_cas.mol, dm=fo_dm)

        g = vj - 0.5 * vk
        g_energy = np.einsum("ij,ij->", g, dm)
        return g_energy

    mydft = dft.UKS(mol)
    mydft.xc = 'b3lyp'

    es_dm = get_es_dm(mydmet, es_mf)
    es_dm_full = get_es_dm(mydmet, es_mf, 1)

    e_dft_corr = mydft.energy_tot(es_dm_full) - mydft.energy_tot(es_dm)
    e_g = get_e_g(mydmet, es_dm)

    e_CL = es_mp.e_tot+mf.energy_nuc()+e_dft_corr-e_g
    print(f"e_CL{i}:", e_CL)
    energy.append(e_CL)
print(energy)



System: uname_result(system='Linux', node='localhost', release='4.4.0-26100-Microsoft', version='#8737-Microsoft Fri Jan 01 08:00:00 PST 2016', machine='x86_64')  Threads 8
Python 3.10.12 (main, Feb  4 2025, 14:57:36) [GCC 11.4.0]
numpy 1.24.2  scipy 1.15.3  h5py 3.13.0
Date: Sat Aug 15 15:47:50 2026
PySCF version 2.9.0
PySCF path  /home/soda/.local/lib/python3.10/site-packages/pyscf

[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 9
[INPUT] num. electrons = 97
[INPUT] charge = -2
[INPUT] spin (= nelec alpha-beta = 2S) = 3
[INPUT] symmetry 0 subgroup None
[INPUT] Mole.unit = angstrom
[INPUT] Symbol           X                Y                Z      unit          X                Y                Z       unit  Magmom
[INPUT]  1 Co     0.000000000000   0.000000000000   0.000000000000 AA    0.000000000000   0.000000000000   0.000000000000 Bohr   0.0
[INPUT]  2 S      2.301865900000   0.000000000000   0.000000000000 AA    4.349896126475   0.000000000000   0.000000000000 Bo

In [7]:
B0 = np.array(energy)-np.array(energy_as)
print((B0-B0[0])*27.2113863)
print((B0-B0[0])*27.2113863*23.06)

[ 0.00000000e+00  1.54164267e-01 -1.80765408e-02 -1.18680011e-01
  1.45528307e-01  3.19359582e-01 -2.88785937e-01  3.19359490e-01
  1.45528341e-01 -1.18680041e-01 -1.80764938e-02  1.54164172e-01
  4.53147080e-08]
[ 0.00000000e+00  3.55502799e+00 -4.16845032e-01 -2.73676106e+00
  3.35588275e+00  7.36443195e+00 -6.65940370e+00  7.36442984e+00
  3.35588355e+00 -2.73676176e+00 -4.16843948e-01  3.55502582e+00
  1.04495717e-06]


In [8]:
print(np.array(energy))
print(np.array(energy_as))

[-2981.59040416 -2981.58682712 -2981.59257314 -2981.59656766
 -2981.59075882 -2981.58330417 -2981.60039043 -2981.58330418
 -2981.59075882 -2981.59656766 -2981.59257314 -2981.58682713
 -2981.59040416]
[-2987.31733151 -2987.31941991 -2987.3188362  -2987.3191336
 -2987.32303424 -2987.32196778 -2987.3167051  -2987.32196778
 -2987.32303424 -2987.3191336  -2987.3188362  -2987.31941991
 -2987.31733151]
